In [64]:
-- 03_gold_model - Cell 1: dim_date (calendar dimension)
CREATE OR REPLACE TABLE dim_date
USING DELTA
AS
WITH date_range AS (
    SELECT explode(sequence(
        to_date('2022-01-01'),
        to_date('2024-12-31'),
        interval 1 day
    )) AS full_date
)
SELECT
    CAST(date_format(full_date, 'yyyyMMdd') AS INT)  AS date_key,
    full_date,
    YEAR(full_date)                                    AS year,
    QUARTER(full_date)                                 AS quarter,
    MONTH(full_date)                                   AS month,
    DATE_FORMAT(full_date, 'MMMM')                     AS month_name,
    DATE_FORMAT(full_date, 'MMM')                      AS month_short,
    WEEKOFYEAR(full_date)                              AS week_iso,
    DAY(full_date)                                     AS day_of_month,
    DAYOFWEEK(full_date)                               AS day_of_week,
    DATE_FORMAT(full_date, 'EEEE')                     AS day_name,
    DATE_FORMAT(full_date, 'EEE')                      AS day_short,
    CASE WHEN DAYOFWEEK(full_date) = 1
         THEN 7 ELSE DAYOFWEEK(full_date) - 1 END      AS day_order_mon,
    CASE WHEN DAYOFWEEK(full_date) IN (1, 7) 
         THEN 1 ELSE 0 END                             AS is_weekend,
    CASE
        WHEN MONTH(full_date) IN (12, 1, 2)  THEN 'Winter'
        WHEN MONTH(full_date) IN (3, 4, 5)   THEN 'Spring'
        WHEN MONTH(full_date) IN (6, 7, 8)   THEN 'Summer'
        ELSE 'Fall'
    END                                                AS season,
    CASE
        WHEN MONTH(full_date) IN (3, 4, 5)   THEN 1
        WHEN MONTH(full_date) IN (6, 7, 8)   THEN 2
        WHEN MONTH(full_date) IN (9, 10, 11) THEN 3
        WHEN MONTH(full_date) IN (12, 1, 2)  THEN 4
    END                                                AS season_order,
    CONCAT(YEAR(full_date), '-Q', QUARTER(full_date)) AS year_quarter,
    CONCAT(YEAR(full_date), '-', LPAD(CAST(MONTH(full_date) AS STRING), 2, '0')) AS year_month
FROM date_range
ORDER BY full_date;

StatementMeta(, 7d3bea28-bb2b-47d8-b298-054a2f9741c2, 72, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [65]:
-- 03_gold_model - Cell 2: dim_time (hour-of-day dimension)
CREATE OR REPLACE TABLE dim_time
USING DELTA
AS
WITH hours AS (
    SELECT explode(sequence(0, 23, 1)) AS hour
)
SELECT
    hour                                                AS time_key,
    hour,
    LPAD(CAST(hour AS STRING), 2, '0') || ':00'         AS hour_label,
    CASE
        WHEN hour BETWEEN 0  AND 5  THEN 'Late Night'
        WHEN hour BETWEEN 6  AND 11 THEN 'Morning'
        WHEN hour BETWEEN 12 AND 17 THEN 'Afternoon'
        ELSE 'Evening'
    END                                                 AS day_part,
    CASE
        WHEN hour BETWEEN 0  AND 5  THEN 1
        WHEN hour BETWEEN 6  AND 11 THEN 2
        WHEN hour BETWEEN 12 AND 17 THEN 3
        ELSE 4
    END                                                 AS day_part_order,
    CASE
        WHEN hour BETWEEN 6  AND 10  THEN 1
        WHEN hour BETWEEN 16 AND 20 THEN 1
        ELSE 0
    END                                                 AS is_rush_hour
FROM hours
ORDER BY hour;

StatementMeta(, 7d3bea28-bb2b-47d8-b298-054a2f9741c2, 73, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [66]:
-- 03_gold_model - Cell 3: dim_vendor (TPEP provider lookup)
CREATE OR REPLACE TABLE dim_vendor
USING DELTA
AS
SELECT * FROM VALUES
    (1, 1,    'Creative Mobile Technologies'),
    (2, 2,    'Curb Mobility'),
    (3, 6,    'Myle Technologies'),
    (4, 7,    'Helix'),
    (5, NULL, 'Unknown')
AS dim_vendor(vendor_key, vendor_id, vendor_name);

StatementMeta(, 7d3bea28-bb2b-47d8-b298-054a2f9741c2, 74, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [67]:
-- 03_gold_model - Cell 4: dim_ratecode (TLC rate code lookup)
CREATE OR REPLACE TABLE dim_ratecode
USING DELTA
AS
SELECT * FROM VALUES
    (1, 1,    'Standard Rate'),
    (2, 2,    'JFK'),
    (3, 3,    'Newark'),
    (4, 4,    'Nassau or Westchester'),
    (5, 5,    'Negotiated Fare'),
    (6, 6,    'Group Ride'),
    (7, 99,   'Null/Unknown')
AS dim_ratecode(ratecode_key, ratecode_id, ratecode_name);

StatementMeta(, 7d3bea28-bb2b-47d8-b298-054a2f9741c2, 75, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [68]:
-- 03_gold_model - Cell 5: dim_payment (payment type lookup)
CREATE OR REPLACE TABLE dim_payment
USING DELTA
AS
SELECT * FROM VALUES
    (1, 0, 'Flex Fare Trip'),
    (2, 1, 'Credit Card'),
    (3, 2, 'Cash'),
    (4, 3, 'No Charge'),
    (5, 4, 'Dispute'),
    (6, 5, 'Unknown'),
    (7, 6, 'Voided Trip')
AS dim_payment(payment_key, payment_type_id, payment_type_name);

StatementMeta(, 7d3bea28-bb2b-47d8-b298-054a2f9741c2, 76, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [69]:
-- 03_gold_model - Cell 6: validate dimensions A
SELECT 'dim_date'      AS table_name, COUNT(*) AS rows FROM dim_date
UNION ALL
SELECT 'dim_time'      AS table_name, COUNT(*) AS rows FROM dim_time
UNION ALL
SELECT 'dim_vendor'    AS table_name, COUNT(*) AS rows FROM dim_vendor
UNION ALL
SELECT 'dim_ratecode'  AS table_name, COUNT(*) AS rows FROM dim_ratecode
UNION ALL
SELECT 'dim_payment'   AS table_name, COUNT(*) AS rows FROM dim_payment
ORDER BY table_name;

StatementMeta(, 7d3bea28-bb2b-47d8-b298-054a2f9741c2, 77, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 2 fields>

In [70]:
%%pyspark
# 03_gold_model - Cell 7: Borough-level ACS demographics (snapshot)
# Source: U.S. Census Bureau, American Community Survey 2022 5-year estimates
# Reference data — refresh yearly in production
# Production alternative: NYC Open Data SODA API via sodapy (commented at bottom)

from pyspark.sql import Row

# ACS 2022 5-year estimates by Borough + Newark (EWR area)
acs_data = [
    Row(borough="Manhattan",     total_population=1629153, median_household_income=93956, area_sq_mi=22.83),
    Row(borough="Brooklyn",      total_population=2561225, median_household_income=74692, area_sq_mi=70.82),
    Row(borough="Queens",        total_population=2252196, median_household_income=79567, area_sq_mi=108.53),
    Row(borough="Bronx",         total_population=1356476, median_household_income=46838, area_sq_mi=42.10),
    Row(borough="Staten Island", total_population=490687,  median_household_income=99210, area_sq_mi=58.37),
    Row(borough="EWR",           total_population=311549,  median_household_income=42553, area_sq_mi=26.11),
    Row(borough="Unknown",       total_population=None,    median_household_income=None,  area_sq_mi=None),
]

df_acs = spark.createDataFrame(acs_data)

# Derive population density (people per square mile)
from pyspark.sql import functions as F
df_acs = df_acs.withColumn(
    "population_density_per_sq_mi",
    F.when(F.col("area_sq_mi") > 0,
           F.col("total_population") / F.col("area_sq_mi"))
     .otherwise(None)
)

df_acs.show(truncate=False)

# Persist as Delta
(df_acs.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_acs_borough_demographics"))

print("Table gold_acs_borough_demographics persisted.")

# -----------------------------------------------------------------
# PRODUCTION ALTERNATIVE: fetch from NYC Open Data via SODA API
# Uncomment and adapt if pulling live data is preferred.
#
# from sodapy import Socrata
# client = Socrata("data.cityofnewyork.us", NYC_APP_TOKEN)
# results = client.get("kku6-nxdu", limit=300)
# df_acs_api = spark.createDataFrame(results)
# -----------------------------------------------------------------

StatementMeta(, 7d3bea28-bb2b-47d8-b298-054a2f9741c2, 78, Finished, Available, Finished, False)

+-------------+----------------+-----------------------+----------+----------------------------+
|borough      |total_population|median_household_income|area_sq_mi|population_density_per_sq_mi|
+-------------+----------------+-----------------------+----------+----------------------------+
|Manhattan    |1629153         |93956                  |22.83     |71360.18396846256           |
|Brooklyn     |2561225         |74692                  |70.82     |36165.278170008474          |
|Queens       |2252196         |79567                  |108.53    |20751.82898737676           |
|Bronx        |1356476         |46838                  |42.1      |32220.332541567695          |
|Staten Island|490687          |99210                  |58.37     |8406.493061504198           |
|EWR          |311549          |42553                  |26.11     |11932.171581769437          |
|Unknown      |NULL            |NULL                   |NULL      |NULL                        |
+-------------+---------------

StatementMeta(, 7d3bea28-bb2b-47d8-b298-054a2f9741c2, 79, Finished, Available, Finished, False)


=== Fetching 2022 ===
  offset=    1  fetched=1000  total_year=1000
  offset= 1001  fetched=825  total_year=1825

=== Fetching 2023 ===
  offset=    1  fetched=1000  total_year=1000
  offset= 1001  fetched=825  total_year=1825

=== Fetching 2024 ===
  offset=    1  fetched=1000  total_year=1000
  offset= 1001  fetched=830  total_year=1830

=== Total raw records across all years: 5480 ===

Long format sample:
+----------+--------+-------------------+-----------------+-----+
|attributes|datatype|date               |station          |value|
+----------+--------+-------------------+-----------------+-----+
|,,W,2400  |PRCP    |2022-01-01T00:00:00|GHCND:USW00094728|20.1 |
|,,W,      |SNOW    |2022-01-01T00:00:00|GHCND:USW00094728|0.0  |
|,,W,2400  |SNWD    |2022-01-01T00:00:00|GHCND:USW00094728|0.0  |
|,,W,2400  |TMAX    |2022-01-01T00:00:00|GHCND:USW00094728|13.3 |
|,,W,2400  |TMIN    |2022-01-01T00:00:00|GHCND:USW00094728|10.0 |
+----------+--------+-------------------+-----------------+

In [72]:
-- 03_gold_model - Cell 9: validate external data tables
SELECT 'gold_acs_borough_demographics' AS table_name, COUNT(*) AS rows FROM gold_acs_borough_demographics
UNION ALL
SELECT 'gold_noaa_weather'             AS table_name, COUNT(*) AS rows FROM gold_noaa_weather
ORDER BY table_name;

StatementMeta(, 7d3bea28-bb2b-47d8-b298-054a2f9741c2, 80, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

In [73]:
-- 03_gold_model - Cell 10: enrich dim_date with NOAA weather
CREATE OR REPLACE TABLE dim_date
USING DELTA
AS
WITH date_range AS (
    SELECT explode(sequence(
        to_date('2022-01-01'),
        to_date('2024-12-31'),
        interval 1 day
    )) AS full_date
)
SELECT
    CAST(date_format(d.full_date, 'yyyyMMdd') AS INT)         AS date_key,
    d.full_date,
    YEAR(d.full_date)                                          AS year,
    QUARTER(d.full_date)                                       AS quarter,
    MONTH(d.full_date)                                         AS month,
    DATE_FORMAT(d.full_date, 'MMMM')                           AS month_name,
    DATE_FORMAT(d.full_date, 'MMM')                            AS month_short,
    WEEKOFYEAR(d.full_date)                                    AS week_iso,
    DAY(d.full_date)                                           AS day_of_month,
    DAYOFWEEK(d.full_date)                                     AS day_of_week,
    DATE_FORMAT(d.full_date, 'EEEE')                           AS day_name,
    DATE_FORMAT(d.full_date, 'EEE')                            AS day_short,
    CASE WHEN DAYOFWEEK(d.full_date) = 1
         THEN 7 ELSE DAYOFWEEK(d.full_date) - 1 END            AS day_order_mon,
    CASE WHEN DAYOFWEEK(d.full_date) IN (1, 7) THEN 1 ELSE 0 END AS is_weekend,
    CASE
        WHEN MONTH(d.full_date) IN (12, 1, 2)  THEN 'Winter'
        WHEN MONTH(d.full_date) IN (3, 4, 5)   THEN 'Spring'
        WHEN MONTH(d.full_date) IN (6, 7, 8)   THEN 'Summer'
        ELSE 'Fall'
    END                                                        AS season,
    CASE
        WHEN MONTH(d.full_date) IN (3, 4, 5)   THEN 1
        WHEN MONTH(d.full_date) IN (6, 7, 8)   THEN 2
        WHEN MONTH(d.full_date) IN (9, 10, 11) THEN 3
        WHEN MONTH(d.full_date) IN (12, 1, 2)  THEN 4
    END                                                        AS season_order,
    CONCAT(YEAR(d.full_date), '-Q', QUARTER(d.full_date))     AS year_quarter,
    CONCAT(YEAR(d.full_date), '-', 
           LPAD(CAST(MONTH(d.full_date) AS STRING), 2, '0'))   AS year_month,
    -- Weather enrichment (NOAA)
    w.temp_max_celsius_tenths,
    w.temp_min_celsius_tenths,
    w.precipitation_mm_tenths,
    w.snowfall_mm,
    w.snow_depth_mm,
    COALESCE(w.weather_category, 'Unknown')                    AS weather_category,
    COALESCE(w.is_extreme_weather, 0)                          AS is_extreme_weather
FROM date_range d
LEFT JOIN gold_noaa_weather w ON w.full_date = d.full_date
ORDER BY d.full_date;

StatementMeta(, 7d3bea28-bb2b-47d8-b298-054a2f9741c2, 81, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [74]:
-- 03_gold_model - Cell 11: dim_zone enriched with ACS borough-level demographics
CREATE OR REPLACE TABLE dim_zone
USING DELTA
AS
SELECT
    ROW_NUMBER() OVER (ORDER BY z.LocationID)             AS zone_key,
    z.LocationID                                          AS location_id,
    z.Borough                                             AS borough,
    z.Zone                                                AS zone_name,
    z.service_zone                                        AS service_zone,
    COALESCE(a.total_population,            0)            AS total_population,
    COALESCE(a.median_household_income,     0)            AS median_household_income,
    COALESCE(a.area_sq_mi,                  0)            AS area_sq_mi,
    COALESCE(a.population_density_per_sq_mi, 0)           AS population_density_per_sq_mi
FROM gold_taxi_zone_lookup z
LEFT JOIN gold_acs_borough_demographics a
       ON UPPER(TRIM(a.borough)) = UPPER(TRIM(z.Borough))
ORDER BY z.LocationID;

StatementMeta(, 7d3bea28-bb2b-47d8-b298-054a2f9741c2, 82, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [75]:
-- 03_gold_model - Cell 12: fact_trips with surrogate keys to all dimensions
CREATE OR REPLACE TABLE fact_trips
USING DELTA
PARTITIONED BY (year, month)
AS
SELECT
    -- Surrogate keys to dimensions
    CAST(date_format(s.pickup_datetime, 'yyyyMMdd') AS INT)   AS date_key,
    s.pickup_hour                                              AS time_key,
    zpu.zone_key                                               AS pu_zone_key,
    zdo.zone_key                                               AS do_zone_key,
    COALESCE(dv.vendor_key,   5)                               AS vendor_key,
    COALESCE(dr.ratecode_key, 7)                               AS ratecode_key,
    COALESCE(dp.payment_key,  6)                               AS payment_key,
    -- Partition columns
    s.year,
    s.month,
    -- Measures
    1                                                          AS trip_count,
    s.passenger_count,
    s.trip_distance_mi,
    s.duration_min,
    -- Duration bucket (mirrors ClassifyTripDuration UDF: 5/15/30/60/180 min)
    CASE
        WHEN s.duration_min IS NULL THEN '0 · Unknown'
        WHEN s.duration_min < 5     THEN '1 · Very Short'
        WHEN s.duration_min < 15    THEN '2 · Short'
        WHEN s.duration_min < 30    THEN '3 · Medium'
        WHEN s.duration_min < 60    THEN '4 · Long'
        WHEN s.duration_min < 180   THEN '5 · Very Long'
        ELSE '6 · Outlier'
    END                                                        AS duration_class,
    s.avg_speed_mph,
    s.fare_amount,
    s.extra_amount,
    s.mta_tax,
    s.tip_amount,
    s.tolls_amount,
    s.improvement_surcharge,
    s.total_amount,
    s.congestion_surcharge,
    s.airport_fee,
    -- Flags
    s.is_anomalous,
    s.is_weekend,
    s.day_part
FROM silver_yellow_trips_clean s
LEFT JOIN dim_zone     zpu ON zpu.location_id    = s.pu_location_id
LEFT JOIN dim_zone     zdo ON zdo.location_id    = s.do_location_id
LEFT JOIN dim_vendor   dv  ON dv.vendor_id       = s.vendor_id
LEFT JOIN dim_ratecode dr  ON dr.ratecode_id     = s.ratecode_id
LEFT JOIN dim_payment  dp  ON dp.payment_type_id = s.payment_type_id;

StatementMeta(, 7d3bea28-bb2b-47d8-b298-054a2f9741c2, 83, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [76]:
-- 03_gold_model - Cell 13: gold layer validation
SELECT 'dim_date'      AS table_name, COUNT(*) AS rows FROM dim_date
UNION ALL
SELECT 'dim_time'      AS table_name, COUNT(*) AS rows FROM dim_time
UNION ALL
SELECT 'dim_zone'      AS table_name, COUNT(*) AS rows FROM dim_zone
UNION ALL
SELECT 'dim_vendor'    AS table_name, COUNT(*) AS rows FROM dim_vendor
UNION ALL
SELECT 'dim_ratecode'  AS table_name, COUNT(*) AS rows FROM dim_ratecode
UNION ALL
SELECT 'dim_payment'   AS table_name, COUNT(*) AS rows FROM dim_payment
UNION ALL
SELECT 'fact_trips'    AS table_name, COUNT(*) AS rows FROM fact_trips
ORDER BY table_name;

StatementMeta(, 7d3bea28-bb2b-47d8-b298-054a2f9741c2, 84, Finished, Available, Finished, False)

<Spark SQL result set with 7 rows and 2 fields>

In [77]:
-- 03_gold_model - Cell 14: detect any orphan keys in fact_trips
SELECT
    SUM(CASE WHEN date_key      IS NULL THEN 1 ELSE 0 END) AS null_date_keys,
    SUM(CASE WHEN time_key      IS NULL THEN 1 ELSE 0 END) AS null_time_keys,
    SUM(CASE WHEN pu_zone_key   IS NULL THEN 1 ELSE 0 END) AS null_pu_zone_keys,
    SUM(CASE WHEN do_zone_key   IS NULL THEN 1 ELSE 0 END) AS null_do_zone_keys,
    SUM(CASE WHEN vendor_key    IS NULL THEN 1 ELSE 0 END) AS null_vendor_keys,
    SUM(CASE WHEN ratecode_key  IS NULL THEN 1 ELSE 0 END) AS null_ratecode_keys,
    SUM(CASE WHEN payment_key   IS NULL THEN 1 ELSE 0 END) AS null_payment_keys
FROM fact_trips;

StatementMeta(, 7d3bea28-bb2b-47d8-b298-054a2f9741c2, 85, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 7 fields>

In [78]:
-- BUSINESS SANITY QUERY: receita por borough x year
SELECT
    z.borough,
    d.year,
    COUNT(*)                                   AS trips,
    ROUND(SUM(f.total_amount), 0)              AS total_revenue_usd,
    ROUND(AVG(f.total_amount), 2)              AS avg_fare,
    ROUND(AVG(f.duration_min), 1)              AS avg_duration_min,
    ROUND(AVG(f.trip_distance_mi), 2)          AS avg_distance_mi
FROM fact_trips f
JOIN dim_zone z ON z.zone_key = f.pu_zone_key
JOIN dim_date d ON d.date_key = f.date_key
GROUP BY z.borough, d.year
ORDER BY z.borough, d.year;

StatementMeta(, 7d3bea28-bb2b-47d8-b298-054a2f9741c2, 86, Finished, Available, Finished, False)

<Spark SQL result set with 24 rows and 7 fields>

In [79]:
CREATE OR REPLACE TABLE dim_zone_dropoff
USING DELTA
AS
SELECT
    zone_key                      AS dropoff_zone_key,
    location_id                   AS dropoff_location_id,
    borough                       AS dropoff_borough,
    zone_name                     AS dropoff_zone_name,
    service_zone                  AS dropoff_service_zone,
    total_population              AS dropoff_total_population,
    median_household_income       AS dropoff_median_household_income,
    area_sq_mi                    AS dropoff_area_sq_mi,
    population_density_per_sq_mi  AS dropoff_population_density
FROM dim_zone;

StatementMeta(, 7d3bea28-bb2b-47d8-b298-054a2f9741c2, 87, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
ALTER TABLE dim_date ADD COLUMNS (day_order_mon INT);
UPDATE dim_date
SET day_order_mon = CASE WHEN day_of_week = 1 THEN 7 ELSE day_of_week - 1 END;

StatementMeta(, 48ad0fd1-878d-4fbc-a847-b46a33f87658, 2, Submitted, Cancelling, Cancelling, True)

In [ ]:
-- 1) bate o total (tem que voltar 115.756.175)
SELECT COUNT(*) FROM fact_trips;

-- 2) confere os buckets (massa em Short/Medium, Outlier pequeno, idealmente nada em '0 · Unknown')
SELECT duration_class, COUNT(*) AS trips
FROM fact_trips
GROUP BY duration_class
ORDER BY duration_class;

StatementMeta(, 7d3bea28-bb2b-47d8-b298-054a2f9741c2, -1, Cancelled, , Cancelled, True)

In [ ]:
CREATE OR REPLACE TABLE discard_reasons USING DELTA AS
SELECT * FROM VALUES
  ('Distance <= 0', 2123821),
  ('Fare < 0', 1365574),
  ('Dropoff <= Pickup', 61114),
  ('Temporal leak', 663)
AS t(reason, rows);

StatementMeta(, 7d3bea28-bb2b-47d8-b298-054a2f9741c2, -1, Cancelled, , Cancelled, True)